<a href="https://colab.research.google.com/github/astronomy-commons/lsdb-foundation-model/blob/main/notebooks/tokenize_legacysurvey.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Tokenizing Legacy Survey images with AION-1

This notebook streams galaxy cutouts from the
[`UniverseTBD/mmu_ssl_legacysurvey_north`](https://huggingface.co/datasets/UniverseTBD/mmu_ssl_legacysurvey_north) Hugging Face dataset
(the northern Legacy Surveys DR9 sample from the Multimodal Universe, 14M objects with
g, r, z cutouts of 152x152 pixels), encodes each image into discrete tokens with the
[AION-1](https://github.com/PolymathicAI/AION) image codec, and writes the result as a
parquet Hugging Face dataset with the **same columns as the original catalog, minus the
images, plus an `image_tokens` column**.

The code below is the content of
[`scripts/tokenize_legacysurvey.py`](https://github.com/astronomy-commons/lsdb-foundation-model/blob/main/scripts/tokenize_legacysurvey.py) in this repository; the notebook is generated from it.

For the demo we only process 100 objects and do not push anything to the Hub.

### Enabling GPU access

The codec runs on CPU too, but on Colab go to `Runtime > Change runtime type` and select a
GPU to speed things up.

### Installing dependencies

In [ ]:
!pip install --quiet --upgrade polymathic-aion datasets huggingface_hub pyarrow

## Step I: the tokenization function

A batch of cutouts is a `(batch, bands, height, width)` array of fluxes in nanomaggies.
We wrap it in AION's `LegacySurveyImage` modality, which tells the codec which survey
and bands the pixels come from (the codec knows how to handle a missing `i` band), and
the `CodecManager` downloads the image codec and turns each cutout into
576 integer tokens (a 24x24 grid over the central 96x96 pixels).

In [ ]:
import json
import time
from collections.abc import Iterable, Iterator
from pathlib import Path

import numpy as np
import pyarrow as pa
import pyarrow.parquet as pq
import torch
from aion.codecs import CodecManager
from aion.modalities import LegacySurveyImage
from datasets import Features, IterableDataset, Value, load_dataset
from tqdm.auto import tqdm

DATASET_ID = "UniverseTBD/mmu_ssl_legacysurvey_north"
IMAGE_COLUMN = "image"
TOKEN_COLUMN = "image_tokens"
NUM_IMAGE_TOKENS = LegacySurveyImage.num_tokens  # 576 = 24 x 24 tokens per cutout

def _default_device() -> str:
    return "cuda" if torch.cuda.is_available() else "cpu"


def _aion_band_name(band: str) -> str:
    """Map the catalog band label (e.g. ``des-g``) to the AION label (``DES-G``)."""
    return band.upper()


def tokenize_images(
    flux: np.ndarray,
    bands: list[str],
    codec_manager: CodecManager,
    device: str | torch.device,
) -> np.ndarray:
    """Encode a batch of cutouts into AION-1 image tokens.

    Args:
        flux: array of shape ``(batch, num_bands, height, width)`` in nanomaggies.
        bands: AION band labels, one per channel (e.g. ``["DES-G", "DES-R", "DES-Z"]``).
        codec_manager: an AION ``CodecManager``; the image codec is downloaded on first use.
        device: device on which the codec runs.

    Returns:
        int64 array of shape ``(batch, NUM_IMAGE_TOKENS)``.
    """
    image = LegacySurveyImage(
        flux=torch.as_tensor(np.ascontiguousarray(flux, dtype=np.float32), device=device),
        bands=list(bands),
    )
    tokens = codec_manager.encode(image)[LegacySurveyImage.token_key]
    return tokens.detach().cpu().numpy().astype(np.int64)


## Step II: streaming the catalog

The dataset is 3.4 TiB, so we open it in streaming mode: only the parquet row groups we
actually consume get downloaded. Each streamed batch is a column-oriented dict; we drop
the `image` column, tokenize it, and pass every other column through untouched.

In [ ]:
def open_dataset(dataset_id: str = DATASET_ID, split: str = "train") -> IterableDataset:
    """Open the catalog in streaming mode, so only the rows we consume are downloaded."""
    ds = load_dataset(dataset_id, split=split, streaming=True)
    if ds.features is None:  # features are resolved lazily for streaming parquet datasets
        ds = ds._resolve_features()
    return ds


def tokenize_batch(
    batch: dict[str, list],
    codec_manager: CodecManager,
    device: str | torch.device,
) -> dict[str, list]:
    """Turn one batch of catalog rows into rows with tokens instead of images.

    ``batch`` is a column-oriented dict as produced by ``IterableDataset.batch``.
    Every column except ``image`` is passed through unchanged.
    """
    images = batch[IMAGE_COLUMN]
    bands = [_aion_band_name(b) for b in images[0]["band"]]
    if any([_aion_band_name(b) for b in im["band"]] != bands for im in images):
        raise ValueError("All images in a batch must share the same band ordering")

    flux = np.stack([np.asarray(im["flux"], dtype=np.float32) for im in images])
    tokens = tokenize_images(flux, bands, codec_manager, device)

    out = {key: value for key, value in batch.items() if key != IMAGE_COLUMN}
    out[TOKEN_COLUMN] = tokens
    return out


def tokenize_dataset(
    ds: IterableDataset,
    codec_manager: CodecManager,
    batch_size: int = 32,
    max_objects: int | None = None,
    device: str | torch.device | None = None,
) -> Iterator[dict[str, list]]:
    """Yield tokenized batches (column-oriented dicts) from a streaming dataset."""
    device = device or _default_device()
    if max_objects is not None:
        ds = ds.take(max_objects)
    total = None if max_objects is None else -(-max_objects // batch_size)
    for batch in tqdm(ds.batch(batch_size), total=total, desc="Tokenizing", unit="batch"):
        yield tokenize_batch(batch, codec_manager, device)


## Step III: writing a parquet Hugging Face dataset

The output features are the input ones without `image`, plus `image_tokens`. The features
are embedded in the parquet metadata so `load_dataset` recovers the exact schema.

In [ ]:
def output_features(input_features: Features) -> Features:
    """Features of the tokenized dataset: the input ones minus ``image``, plus ``image_tokens``."""
    features = {k: v for k, v in input_features.items() if k != IMAGE_COLUMN}
    features[TOKEN_COLUMN] = [Value("int64")]
    return Features(features)


def _batch_to_table(batch: dict[str, list], schema: pa.Schema) -> pa.Table:
    columns = {}
    for name in schema.names:
        value = batch[name]
        if name == TOKEN_COLUMN:
            tokens = np.asarray(value, dtype=np.int64)
            offsets = np.arange(0, tokens.size + 1, tokens.shape[1], dtype=np.int32)
            columns[name] = pa.ListArray.from_arrays(pa.array(offsets), pa.array(tokens.ravel()))
        else:
            columns[name] = pa.array(value, type=schema.field(name).type)
    return pa.Table.from_pydict(columns, schema=schema)


def write_parquet(
    batches: Iterable[dict[str, list]],
    features: Features,
    output_dir: str | Path,
    filename: str = "train-00000-of-00001.parquet",
) -> Path:
    """Write tokenized batches to a parquet file readable by ``datasets.load_dataset``.

    The Hugging Face ``features`` are embedded in the parquet metadata so that the
    dataset schema round-trips exactly.
    """
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    path = output_dir / filename

    schema = features.arrow_schema.with_metadata(
        {"huggingface": json.dumps({"info": {"features": features.to_dict()}})}
    )
    n_rows = 0
    with pq.ParquetWriter(path, schema=schema) as writer:
        for batch in batches:
            table = _batch_to_table(batch, schema)
            writer.write_table(table)
            n_rows += table.num_rows
    print(f"Wrote {n_rows} rows to {path}")
    return path


## Step IV: run it on 100 objects

In [ ]:
MAX_OBJECTS = 100
BATCH_SIZE = 32
OUTPUT_DIR = "tokenized_demo"

device = _default_device()
print(f"Running image codec on {device}")

codec_manager = CodecManager(device=device)
ds = open_dataset(DATASET_ID)
features = output_features(ds.features)
print(features)

start = time.time()
batches = tokenize_dataset(ds, codec_manager, batch_size=BATCH_SIZE, max_objects=MAX_OBJECTS, device=device)
path = write_parquet(batches, features, OUTPUT_DIR)
print(f"Done in {time.time() - start:.1f}s")

## Step V: check the result

Reload the parquet file as a regular Hugging Face dataset and look at a row.

In [ ]:
tokenized = load_dataset("parquet", data_files=str(path), split="train")
print(tokenized)
row = tokenized[0]
print({k: v for k, v in row.items() if k != TOKEN_COLUMN})
print("tokens:", len(row[TOKEN_COLUMN]), row[TOKEN_COLUMN][:16], "...")

The tokens can be decoded back into a 96x96 cutout with the same codec, which is a
useful sanity check of what information the tokenization retains.

In [ ]:
import matplotlib.pyplot as plt

bands = ["DES-G", "DES-R", "DES-Z"]
tokens = torch.as_tensor(np.asarray(tokenized[:4][TOKEN_COLUMN]), device=device)
reconstructed = codec_manager.decode({LegacySurveyImage.token_key: tokens}, LegacySurveyImage, bands=bands)

fig, axes = plt.subplots(2, 4, figsize=(12, 6))
for k in range(4):
    original = next(iter(ds.skip(k).take(1)))["image"]["flux"]
    axes[0, k].imshow(np.asarray(original)[1, 28:-28, 28:-28], cmap="gray")
    axes[0, k].set_title(f"{tokenized[k]['object_id']} (r, input)")
    axes[1, k].imshow(reconstructed.flux[k, 1].cpu().numpy(), cmap="gray")
    axes[1, k].set_title("decoded from tokens")
for ax in axes.ravel():
    ax.axis("off")

## Next steps

- Remove `max_objects` to tokenize the full catalog (14M objects; use a GPU and a large
  batch size, and shard the output).
- Push the parquet directory to the Hub with `huggingface_hub.HfApi().upload_folder`.